In [9]:
import pandas as pd
import matplotlib as plt
import numpy as np
from ftfy import fix_text

df = pd.read_csv(
    r'/Users/urielle.zach/Data-Preprocessing/globalmart_dirty_orders_2000.csv', 
    keep_default_na=False,
    na_values=['-', '#N/A', 'N/A', 'NULL', '?', 'unknown', 'NONE', 'na', 'None', 'none', 'NaN', 'NA', 'nan', ''], # catch all null values before parsing
    encoding='utf-8'
)   

df.index = df.index + 2 ## shifted index to align with csv

columns = [
    'record_id', 'source_system', 'customer_id', 'customer_name', 
    'customer_email', 'phone_raw', 'age_raw', 'gender_raw', 
    'signup_date_raw', 'order_id', 'order_date_raw', 'ship_date_raw', 
    'delivery_date_raw', 'customer_timezone', 'country_raw', 'city_raw', 
    'postal_code_raw', 'product_category_raw', 'product_name_raw', 
    'sku_raw', 'quantity_raw', 'unit_price_raw', 'currency_raw', 
    'discount_raw', 'shipping_cost_raw', 'item_weight_raw', 
    'payment_method_raw', 'order_status_raw', 'returned_flag_raw', 
    'return_reason_raw', 'satisfaction_score_raw', 'loyalty_points_raw', 
    'site_visits_last30_raw', 'support_tickets_90d_raw', 
    'distance_to_warehouse_km_raw', 'campaign_source_raw', 
    'customer_segment_raw', 'support_ticket_date_raw', 
    'complaint_text_raw', 'ingestion_batch', 'data_source_note'
]

In [10]:
## PART 1
def returnRows(df):
    return f"Total row count: {len(df)}"

def returnDatatype(df):
    return df.dtypes

def checkMissingValues(df, columns):
    null_rows = df[columns].isnull().sum() ## returns how many null rows are there in the series
    missing_values = null_rows[null_rows > 0] ## filters all boolean values that are greater than 0 to be passed through {missing_values} variable
    total_rows = len(df)
    
    percentages = (missing_values / total_rows) * 100
    print("--------------------Missing Rows--------------------")
    summary_df = pd.DataFrame({
        'Missing Count': missing_values,
        'Percentage (%)': percentages.round(2)  
    })

    return summary_df

## specifies which columns and row has null values
def specifyWhere(df, columns):
    missing_locations = {}
    for col in columns:
        null_indeces = df[df[col].isnull()].index.tolist()

        if null_indeces:
            missing_locations[col] = null_indeces

    return missing_locations

## if {value} of {column_name} HAS duplicates:
    ## do: 
        ## return the row where the {value} is duplicated
        ## filter all {value} to show all true
        ## show the number of duplicated {value} in a column

def checkSpecificColumnDuplicates(df, column_name):
    duplicate_mask = df.duplicated(subset=[column_name], keep=False) ## checks every value in index to check if {value} has duplicates
    duplicated_rows = df[duplicate_mask] ## passes all duplicates into duplicated_rows
    duplicated_rows = duplicated_rows.dropna(subset=[column_name]) ## drops all values containing {NaN}
    duplicates_clean_view = duplicated_rows[['record_id', column_name]]

    print(f"Total amount of duplicated rows: {len(duplicates_clean_view)}")
    return duplicates_clean_view

def genderRemap(df):
    print("unique:")
    print(f"{df["gender_raw"].unique()}\n")
    clean_source = df["gender_raw"].str.upper().str.strip()

    mapping = {
            r'(?i)^(NON-BINARY|NB)$': 'Non-Binary', 
            r'(?i)^(MALE|M)$': 'Male', 
            r'(?i)^(FEMALE|F)$':'Female', 
            r'(?i)^PREFER NOT TO SAY$': 'Prefer not to say'
            
            }
    
    print(f"dict: {mapping} \n")
    df["gender_raw"] = clean_source.replace(mapping, regex=True)
    
    ## recheck
    print("remapped gender_raw")
    return df["gender_raw"]

def sourceRemap(df):
    print("unique: ")
    print(f"{df["source_system"].unique()}\n")
    clean_source = df["source_system"].str.upper().str.strip()

    mapping = {
        r'(?i)^(STORE|IN-STORE)$': 'In-Store',
        r'(?i)^(MOBILE-APP|MOBILE APP|MOBILE|APP)$': 'Mobile App',
        r'(?i)^(CALL CENTRE|CALL CENTER)$': 'Call Center',
        r'(?i)^(PARTNER-API|PARTNER API)$': 'Partner-API',
        r'(?i)^(MARKETPLACE|MARKET PLACE)$': 'Marketplace',
        r'MARKETPLACE | MARKET PLACE': 'Marketplace',
        r'(?i)^WEB$': 'Web'
    }
    
    print(f"dict: {mapping} \n")
    df["source_system"] = clean_source.replace(mapping, regex=True)

    print("remapped source_system")
    return df["source_system"]

def countryRemap(df): ## did this part earlier to fix all data
    print("unique: ")
    print(df["country_raw"].unique())
    
    clean_source = df["country_raw"].str.upper().str.strip()
    mapping = {
        r'(?i)^AUS?$|^Australia$': 'Australia',
        r'(?i)^DE$|^Ger(many)?$|^Deutschland$': 'Germany',
        r'(?i)^FR(ance|nce)?$': 'France',
        r'(?i)^SG$|^Singapore$|^S\'pore$': 'Singapore',
        r'(?i)^JP$|^Japan$|^Nippon$': 'Japan',
        r'(?i)^CA(nada|nda)?$': 'Canada',
        r'(?i)^BR(asil|azil)?$': 'Brazil',
        r'(?i)^PH(L|ils\.?)?$|^Philippines$|^Pilipinas$': 'Philippines',
        r'(?i)^US(A|\.S\.A\.)?$|^United States$|^America$': 'United States',
        r'(?i)^CH$|^Swi(tzerland|ss)$|^Suisse$': 'Switzerland'
    }
    
    print(f"dict: {mapping} \n")
    df["country_raw"] = clean_source.replace(mapping, regex=True)

    print("remapped country_raw")
    return df["country_raw"]

def detectInvalidAge(df):
    age_numeric = pd.to_numeric(df["age_raw"], errors='coerce')
    invalid_condition = (age_numeric >= 100) | (age_numeric <= 13) | (age_numeric.isna())
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'age_raw']]

def detectInvalidDiscount(df):
    if df["discount_raw"].dtype == '0':
        mapping = {
        'freebie': '100%',
        'ten percent': '10%'
        }
            
        df["discount_raw"] = df["discount_raw"].replace(mapping, regex=True)
        df["discount_raw"] = df["discount_raw"].str.rstrip('%').astype(float)
        
    discounts = pd.to_numeric(df["discount_raw"], errors='coerce')    
    invalid_condition = (discounts <= 0) | (discounts >= 101) | (discounts.isna())
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'discount_raw']]

def invalid_delivery_sequence_flag(df):
    order_date = pd.to_datetime(df["order_date_raw"], format='mixed', errors='coerce')
    delivery_date = pd.to_datetime(df["delivery_date_raw"], format='mixed', errors='coerce')
    invalid_condition = (order_date > delivery_date)
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'order_date_raw', 'delivery_date_raw']]

def invalidEmail(df): ## confirmed no false positives (look at mateo.nez140@example..com) ((double dots))
    regex = r'^[\w.-]+@([\w-]+\.)+[\w-]{2,4}$'
    is_valid = df['customer_email'].str.contains(regex, regex=True, na=False)
    invalid_email = df[~is_valid]
    return invalid_email[['record_id', 'customer_email']]


## email validation regex: ^[\w-\.]+@([\w-]+\.)+[\w-]{2,4}$     

In [11]:
## PART 2 D
# missing categorical contexts
    
def missingCustomerEmail(df):
    missing_customer_email = df[df['customer_email'].isna()]
    return missing_customer_email[['record_id', 'customer_email']]

def missingDeliveryDate(df):
    missing_delivery_date = df[df['delivery_date_raw'].isna()]
    return missing_delivery_date[['record_id', 'delivery_date_raw']]

def missingSupportTicketDate(df):
    missing_support_ticket = df[df['support_ticket_date_raw'].isna()]
    return missing_support_ticket[['record_id', 'support_ticket_date_raw']]

def missingReturnReason(df):
    missing_return_reason = df[df['return_reason_raw'].isna()]
    return missing_return_reason[['record_id', 'return_reason_raw']]





In [12]:
## PART 3
def fixTime(df):
   date_columns = ['signup_date_raw', 'order_date_raw', 'ship_date_raw', 'delivery_date_raw', 'support_ticket_date_raw']
   for i in date_columns:
      parsed_dates = pd.to_datetime(df[i], format="mixed", errors="coerce")
      df[i] = parsed_dates.dt.strftime('%Y-%m-%d')

## FIXME: 
def signupDate(df, record_id):
   return df.loc[df['signup_date_raw']].at[record_id]

def order_datetime(df, date):
   return df.loc[df['order_date_raw'] == date] 

def order_datetime(df, date):
   return df.loc[df['order_date_raw'] == date] 

def support_ticket_datetime(df, date):
   return df.loc[df['support_ticket_date_raw'] == date] 

def days_from_signup_to_order(df, date):
   ## pull signup date of a specific customer
   return 0

def shipping_delay_days(df, date):
   return df.loc[df['order_date_raw'] == date] 

# /FIXME
def fixEncoding(df):
   df['customer_name'] = df['customer_name'].apply(lambda x: fix_text(x) if isinstance(x, str) else x)






In [13]:
## PART 5
def productCategoryNormalization(df):
    print("unique: ")
    print(df["product_category_raw"].unique())
    
    clean_source = df["product_category_raw"].str.upper().str.strip()
    mapping = {
        r'^ELEC.*': 'Electronics', 
        r'^(FASH|APP|CLOTH).*': 'Fashion & Apparel', 
        r'.*(HOME|KITCHEN).*': 'Home & Kitchen', 
        r'.*(BEAUTY|CARE).*': 'Health & Beauty', 
        r'.*(BOOK|READ|MEDIA).*': 'Books & Media', 
        r'.*(SPORT|FITNESS).*': 'Sports & Fitness'
    }
    print(f"dict: {mapping} \n")
    df["product_category_raw"] = clean_source.replace(mapping, regex=True)

def orderStatusNormalization(df):
    print("unique: ")
    print(df["order_status_raw"].unique())
    
    clean_source = df["order_status_raw"].str.upper().str.strip()
    mapping = {
        r'^(DELIV.*|COMPLET.*|OK)$': 'Completed', 
        r'.*(SHIP.*|TRANSIT|WAY).*': 'Shipped', 
        r'.*(PEND.*|HOLD|AWAIT).*': 'Pending', 
        r'^(CANC.*|CNCLD|VOID)$': 'Cancelled', 
        r'.*(FAIL.*|DECLIN.*).*': 'Failed', 
        r'.*(RETURN.*|RMA|REFUND).*': 'Returned/Refunded'
    }
    print(f"dict: {mapping} \n")
    df["order_status_raw"] = clean_source.replace(mapping, regex=True)
    
    


In [ ]:
## PART 6
# FIXME: 
# TODO: all clean datatypes should be merged into a single dataframe
# 
def fixQuantity(df):
    word_mapping = {
        'one': '1', 'two': '2', 'three': '3', 'four': '4', 'five': '5'
    }
    cleaned_text = df["quantity_raw"].replace(word_mapping)
    extracted_numbers = cleaned_text.astype(str).str.extract(r'(-?\d+)', expand=False)
    quantity = pd.to_numeric(extracted_numbers, errors='coerce')
    invalid_condition = (quantity <= 0) | (quantity >= 250)
    df["quantity_clean"] = np.where(invalid_condition, np.nan, quantity)

    return df[['quantity_raw', 'quantity_clean']]

def fixUnitPrice(df):
    # TODO:
    # compare [currency_raw] to [unit_price_raw] to apply valid currency to [unit_price_raw] (optional)
    cleaned = df["unit_price_raw"].astype(str).str.replace(r'[^\d\.,]', '', regex=True)
    cleaned = cleaned.str.replace(r',(\d{1,2})$', r'.\1', regex=True)
    cleaned = cleaned.str.replace(',', '', regex=True)
    unit_price = pd.to_numeric(cleaned, errors='coerce')
    df["unit_price_clean"] = np.where(unit_price <= 0, np.nan, unit_price)
    
    return df[['unit_price_raw', 'unit_price_clean']]

def fixShippingCost(df):
    cleaned = df["shipping_cost_raw"].astype(str).str.replace(r'[^\d\.,]', '', regex=True)
    cleaned = cleaned.str.replace(r',(\d{1,2})$', r'.\1', regex=True)
    cleaned = cleaned.str.replace(',', '', regex=True)
    unit_price = pd.to_numeric(cleaned, errors='coerce')
    df["unit_price_clean"] = np.where(unit_price <= 0, np.nan, unit_price)
    
    return df[['shipping_cost_raw', 'unit_price_clean']]

def fixItemWeight(df):
    # TODO:
    # item weight should normalize between kg and lbs
    cleaned = df["item_weight_raw"].astype(str).str.replace(r'[^\d\.,]', '', regex=True)
    cleaned = cleaned.str.replace(r',(\d{1,2})$', r'.\1', regex=True)
    cleaned = cleaned.str.replace(',', '', regex=True)
    unit_price = pd.to_numeric(cleaned, errors='coerce')
    df["unit_price_clean"] = np.where(unit_price <= 0, np.nan, unit_price)
    
    return df[['item_weight_raw', 'unit_price_clean']]

def fixSatScore(df):
    mapping = {
        'excellent': '5',
        'bad': '1'
    }

    cleaned_text = df["satisfaction_score_raw"].astype(str).str.lower().replace(mapping)
    extracted_scores = cleaned_text.str.extract(r'(\d+)', expand=False)
    scores = pd.to_numeric(extracted_scores, errors='coerce')
    invalid_condition = (scores < 1) | (scores > 5)
    df["satisfaction_score_clean"] = np.where(invalid_condition, np.nan, scores)
    return df[['satisfaction_score_raw', 'satisfaction_score_clean']]

def fixLoyaltyPoints(df):
    points = pd.to_numeric(df["loyalty_points_raw"], errors='coerce')
    invalid_condition = (points < 0) | (points > 100000)
    df["loyalty_points_clean"] = np.where(invalid_condition, np.nan, points)
    df["loyalty_points_clean"] = df["loyalty_points_clean"].astype('Int64')
    return df[['loyalty_points_raw', 'loyalty_points_clean']]

def fixSiteVisits(df):
    siteVisits = pd.to_numeric(df["site_visits_last30_raw"], errors='coerce')
    df["site_visits_last30_clean"] = siteVisits
    df["site_visits_last30_clean"] = df["site_visits_last30_clean"].astype('Int64')
    return df[['site_visits_last30_raw', 'site_visits_last30_clean']]

def fixSupportTickets(df):
    # TODO: fix the 'many' edge cases entries
    supportTickets = pd.to_numeric(df["support_tickets_90d_raw"], errors='coerce')
    df["support_tickets_90d_clean"] = supportTickets
    df["support_tickets_90d_clean"] = df["support_tickets_90d_clean"].astype('Int64')
    return df[['support_tickets_90d_raw', 'support_tickets_90d_clean']]

def fixWarehouseDistance(df):
    warehouseDisance = pd.to_numeric(df["distance_to_warehouse_km_raw"], errors='coerce')
    df["distance_to_warehouse_km_clean"] = warehouseDisance
    df["distance_to_warehouse_km_clean"] = df["distance_to_warehouse_km_clean"].astype('Int64')
    return df[['distance_to_warehouse_km_raw', 'distance_to_warehouse_km_clean']]
# /FIXME

fixWarehouseDistance(df)



,support_tickets_90d_raw,support_tickets_90d_clean
2,0,0
3,3,3
4,3,3
5,1,1
6,1,1
...,...,...
1997,0,0
1998,0,0
1999,1,1
2000,0,0


In [ ]:
# TODO:
# Task 6: Numeric Parsing and Outlier Handling
#Convert messy numeric fields into clean numeric columns.
#Required Fields:


#• distance_to_warehouse_km_raw 

In [16]:
## Clean Everything
def clean(df):
    genderRemap(df)
    sourceRemap(df)
    countryRemap(df)
    fixTime(df)
    fixEncoding(df)
    productCategoryNormalization(df)
    fixQuantity(df)

clean(df)
df['quantity_raw']

unique:
<StringArray>
[       'Non-binary',                 nan,              'male',
                'NB',                 'M',              'Male',
                 'F', 'Prefer not to say',            'female',
            'Female']
Length: 10, dtype: str

dict: {'(?i)^(NON-BINARY|NB)$': 'Non-Binary', '(?i)^(MALE|M)$': 'Male', '(?i)^(FEMALE|F)$': 'Female', '(?i)^PREFER NOT TO SAY$': 'Prefer not to say'} 

remapped gender_raw
unique: 
<StringArray>
[  'Mobile App',          'WEB',         'Web ',     'in-store',
 'Market Place',       'mobile',  'Call Center',  'call centre',
          'web',          'APP',  'marketplace',  'partner-api',
  'Partner API',        'Store',            nan]
Length: 15, dtype: str

dict: {'(?i)^(STORE|IN-STORE)$': 'In-Store', '(?i)^(MOBILE-APP|MOBILE APP|MOBILE|APP)$': 'Mobile App', '(?i)^(CALL CENTRE|CALL CENTER)$': 'Call Center', '(?i)^(PARTNER-API|PARTNER API)$': 'Partner-API', '(?i)^(MARKETPLACE|MARKET PLACE)$': 'Marketplace', 'MARKETPLACE | MARKET P

remapped country_raw
unique: 
<StringArray>
[         'Apparel',            'FASHN',       'Electronic',
     'Electronics ',     'Home/Kitchen',  'Health & Beauty',
          'fashion',          'READING',            'Elec.',
          'Fashion',           'BEAUTY',          'Fitness',
             'Book',         'Clothing',     'HOME-KITCHEN',
     'home kitchen',           'beauty',           'sports',
           'Beauty',             'elec', 'Home and Kitchen',
      'Media/Books',   'Kitchen & Home',   'Sporting Goods',
           'Sports',            'books',          'Clothes',
           'SPORTS',    'Personal Care',   'Home & Kitchen',
      'ELECTRONICS',            'Books',      'Electronics',
                nan]
Length: 34, dtype: str
dict: {'^ELEC.*': 'Electronics', '^(FASH|APP|CLOTH).*': 'Fashion & Apparel', '.*(HOME|KITCHEN).*': 'Home & Kitchen', '.*(BEAUTY|CARE).*': 'Health & Beauty', '.*(BOOK|READ|MEDIA).*': 'Books & Media', '.*(SPORT|FITNESS).*': 'Sports & Fitness'}

2        2
3        1
4        2
5        3
6        1
        ..
1997     1
1998     5
1999    10
2000     1
2001    10
Name: quantity_raw, Length: 2000, dtype: str